In [4]:
# Necessary Imports (Make Sure This Block Runs Before Continuing)
import wfdb
import numpy as np
from collections import Counter

In [5]:
# Taking A Smaller Sample of Six Patients
records = ['100', '101', '102', '103', '104', '105']

In [ ]:
# Stack Overflow (Edited for Our Specifics)
# These will be our outputs from the Random Forest

beat_class_map = {
    'N': 'Normal', 'L': 'Normal', 'R': 'Normal', 'e': 'Normal', 'j': 'Normal',
    'A': 'SVEB', 'a': 'SVEB', 'J': 'SVEB', 'S': 'SVEB',
    'V': 'VEB', 'E': 'VEB',
    'F': 'Fusion',
    '/': 'Unclassifiable', 'f': 'Unclassifiable', 'Q': 'Unclassifiable'
}

In [7]:
# Classifying Specific Segments (Not Individualized)
segments = []
labels = []
window_size = 100

for rec in records:
    record_path = f"data/data/MITBIHRawData/{rec}"
    
    record = wfdb.rdrecord(record_path)
    annotation = wfdb.rdann(record_path, 'atr')
    
    for i in range(len(annotation.sample)):
        symbol = annotation.symbol[i]
        
        if symbol in beat_class_map:
            sample = annotation.sample[i]
            start = sample - window_size
            end = sample + window_size
            
            if start >= 0 and end <= len(record.p_signal):
                segment = record.p_signal[start:end, 0]
                
                if len(segment) == 2 * window_size:
                    segments.append(segment)
                    labels.append(beat_class_map[symbol])

print("Total segments:", len(segments))
print("Class distribution:", Counter(labels))

Total segments: 13206
Class distribution: Counter({'Normal': 8966, 'Unclassifiable': 4154, 'VEB': 48, 'SVEB': 38})


In [ ]:
# Going Ahead and removing unclassifiable to speed up the code.
keep_classes = ['Normal', 'SVEB', 'VEB']

filtered_segments = []
filtered_labels = []

for seg, lab in zip(segments, labels):
    if lab in keep_classes:
        filtered_segments.append(seg)
        filtered_labels.append(lab)

print("Filtered Total Segments:", len(filtered_segments))
print("Filtered Class Distribution:", Counter(filtered_labels))

Filtered total segments: 9052
Filtered class distribution: Counter({'Normal': 8966, 'VEB': 48, 'SVEB': 38})


In [9]:
features = []

for seg in filtered_segments:
    feature_vector = [
        np.mean(seg),
        np.std(seg),
        np.min(seg),
        np.max(seg),
        np.ptp(seg),
        np.sqrt(np.mean(seg**2)),
        np.sum(seg**2)
    ]
    features.append(feature_vector)

X = np.array(features)
y = np.array(filtered_labels)

print("Feature matrix shape:", X.shape)
print("Label array shape:", y.shape)

Feature matrix shape: (9052, 7)
Label array shape: (9052,)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

Training set: (7241, 7) (7241,)
Test set: (1811, 7) (1811,)


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[1794    0    0]
 [   7    0    0]
 [   3    0    7]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00      1794
        SVEB       0.00      0.00      0.00         7
         VEB       1.00      0.70      0.82        10

    accuracy                           0.99      1811
   macro avg       0.66      0.57      0.61      1811
weighted avg       0.99      0.99      0.99      1811



c:\Users\cdduc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\cdduc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\cdduc\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [12]:
patient_summaries = {}

for rec in records:
    record_path = f"data/data/MITBIHRawData/{rec}"
    
    record = wfdb.rdrecord(record_path)
    annotation = wfdb.rdann(record_path, 'atr')
    
    patient_features = []
    
    for i in range(len(annotation.sample)):
        symbol = annotation.symbol[i]
        
        if symbol in beat_class_map:
            mapped_label = beat_class_map[symbol]
            
            if mapped_label in keep_classes:
                sample = annotation.sample[i]
                start = sample - window_size
                end = sample + window_size
                
                if start >= 0 and end <= len(record.p_signal):
                    segment = record.p_signal[start:end, 0]
                    
                    if len(segment) == 2 * window_size:
                        feature_vector = [
                            np.mean(segment),
                            np.std(segment),
                            np.min(segment),
                            np.max(segment),
                            np.ptp(segment),
                            np.sqrt(np.mean(segment**2)),
                            np.sum(segment**2)
                        ]
                        patient_features.append(feature_vector)
    
    if len(patient_features) > 0:
        patient_features = np.array(patient_features)
        patient_preds = rf.predict(patient_features)
        patient_summaries[rec] = Counter(patient_preds)

for rec, summary in patient_summaries.items():
    print(f"Record {rec}: {dict(summary)}")

Record 100: {np.str_('Normal'): 2243, np.str_('SVEB'): 27, np.str_('VEB'): 1}
Record 101: {np.str_('Normal'): 1859, np.str_('SVEB'): 3}
Record 102: {np.str_('Normal'): 99, np.str_('VEB'): 4}
Record 103: {np.str_('Normal'): 2083, np.str_('SVEB'): 1}
Record 104: {np.str_('Normal'): 163, np.str_('VEB'): 2}
Record 105: {np.str_('Normal'): 2529, np.str_('VEB'): 38}


In [13]:
for rec, summary in patient_summaries.items():
    normal_count = summary.get('Normal', 0)
    sveb_count = summary.get('SVEB', 0)
    veb_count = summary.get('VEB', 0)
    fusion_count = summary.get('Fusion', 0)
    unclass_count = summary.get('Unclassifiable', 0)

    print(f"Record {rec}:")
    print(f"  Normal beats: {normal_count}")
    print(f"  SVEB beats: {sveb_count}")
    print(f"  VEB beats: {veb_count}")
    print(f"  Fusion beats: {fusion_count}")
    print(f"  Unclassifiable beats: {unclass_count}")

    interpretation = "This patient is predominantly normal"
    findings = []

    if sveb_count > 0:
        findings.append("shows signs of supraventricular ectopic activity")
    if veb_count > 0:
        findings.append("shows signs of ventricular ectopic activity")
    if fusion_count > 0:
        findings.append("shows signs of fusion beats")

    if findings:
        interpretation += " but " + " and ".join(findings) + "."
    else:
        interpretation += " with no major abnormal beat types detected by the baseline model."

    print("  Interpretation:", interpretation)
    print()

Record 100:
  Normal beats: 2243
  SVEB beats: 27
  VEB beats: 1
  Fusion beats: 0
  Unclassifiable beats: 0
  Interpretation: This patient is predominantly normal but shows signs of supraventricular ectopic activity and shows signs of ventricular ectopic activity.

Record 101:
  Normal beats: 1859
  SVEB beats: 3
  VEB beats: 0
  Fusion beats: 0
  Unclassifiable beats: 0
  Interpretation: This patient is predominantly normal but shows signs of supraventricular ectopic activity.

Record 102:
  Normal beats: 99
  SVEB beats: 0
  VEB beats: 4
  Fusion beats: 0
  Unclassifiable beats: 0
  Interpretation: This patient is predominantly normal but shows signs of ventricular ectopic activity.

Record 103:
  Normal beats: 2083
  SVEB beats: 1
  VEB beats: 0
  Fusion beats: 0
  Unclassifiable beats: 0
  Interpretation: This patient is predominantly normal but shows signs of supraventricular ectopic activity.

Record 104:
  Normal beats: 163
  SVEB beats: 0
  VEB beats: 2
  Fusion beats: 0
  Unc